In [1]:
!pip install gradio requests


In [2]:
!pip install -q transformers

In [3]:
%%writefile grafo_logistica.py
import json

class GrafoLogistica:
    def __init__(self):
        # 1. Tu idea principal: El Caché Vectorial
        # Formato esperado -> "PA123-PA456": {"distancia_km": 4.2, "tiempo_minutos": 12.5, "servicios": ["210", "F30"]}
        self.cache_rutas = {}

        # 2. Diccionario de Adyacencia para la IA
        # Formato esperado -> "PA123": ["PA456", "PA789"] (A quiénes está conectado directamente)
        self.adyacencia = {}

        # 3. Registro único de paraderos (Nodos)
        self.paraderos = set()

    def _generar_llave(self, origen, destino):
        """Método interno privado para asegurar que la llave siempre tenga el mismo formato."""
        return f"{origen}-{destino}"

    def agregar_paradero(self, codigo_paradero):
        """Registra un paradero nuevo en el sistema y le prepara su lista de vecinos."""
        if codigo_paradero not in self.paraderos:
            self.paraderos.add(codigo_paradero)
            self.adyacencia[codigo_paradero] = []

    def registrar_ruta(self, origen, destino, distancia, tiempo, servicios):
        """Guarda los datos devueltos por la API de OSRM en nuestra memoria (Hit de caché)."""
        # 1. Asegurarnos de que los paraderos existan
        self.agregar_paradero(origen)
        self.agregar_paradero(destino)

        # 2. Guardar en el Caché
        llave = self._generar_llave(origen, destino)
        self.cache_rutas[llave] = {
            "distancia_km": distancia,
            "tiempo_minutos": tiempo,
            "servicios": servicios
        }

        # 3. Conectar en la lista de adyacencia (Para que la IA sepa que son vecinos)
        if destino not in self.adyacencia[origen]:
            self.adyacencia[origen].append(destino)

    def consultar_cache(self, origen, destino):
        """
        Le pregunta a la memoria si ya conocemos esta ruta.
        Retorna el diccionario con los vectores de tiempo/distancia o None si no existe.
        """
        llave = self._generar_llave(origen, destino)
        return self.cache_rutas.get(llave, None)

    def obtener_vecinos(self, codigo_paradero):
        """Devuelve la lista de paraderos a los que puedo viajar directamente desde aquí."""
        return self.adyacencia.get(codigo_paradero, [])

    def exportar_cache_json(self, nombre_archivo="cache_rutas.json"):
        """¡Nivel Dios! Guarda el estado actual del caché en un archivo para no perderlo al apagar Colab."""
        with open(nombre_archivo, 'w') as f:
            json.dump(self.cache_rutas, f, indent=4)
        print(f"✅ Caché exportado exitosamente a {nombre_archivo}")

    def mostrar_estado(self):
        """Muestra cuántos datos tiene guardados el grafo actualmente."""
        print("--- ESTADO DEL GRAFO LOGÍSTICO ---")
        print(f"📍 Paraderos registrados: {len(self.paraderos)}")
        print(f"🛣️ Rutas en caché: {len(self.cache_rutas)}")
        print("----------------------------------")

Writing grafo_logistica.py


In [4]:
  %%writefile motor_logico.py
  import requests

  class MotorLogico:
    def __init__(self, grafo):
        # Recibe el objeto GrafoLogistica por inyección de dependencias
        self.grafo = grafo

        # Diccionario para almacenar las coordenadas de los paraderos.
        # Formato: "PA123": (Latitud, Longitud)
        self.coordenadas_paraderos = {}

        # Simulación de una base de datos que nos dice qué micros pasan por qué tramo.
        # Formato: "PA123-PA456": ["210", "F30"]
        self.servicios_tramos = {}

    def cargar_datos_base(self, paraderos_dict, servicios_dict):
        """Simula la carga de un archivo CSV/JSON con las paradas y las micros."""
        self.coordenadas_paraderos = paraderos_dict
        self.servicios_tramos = servicios_dict

    def procesar_ruta(self, origen, destino):
        """
        El corazón de esta clase. Busca en la memoria local; si no está,
        va a internet, lo descarga, lo guarda y lo devuelve.
        """
        # 1. HIT DE CACHÉ: Le preguntamos al grafo si ya conoce el camino
        datos_cacheados = self.grafo.consultar_cache(origen, destino)
        if datos_cacheados:
            print(f"⚡ [Caché Hit] Ruta {origen} -> {destino} cargada desde la memoria al instante.")
            return datos_cacheados

        # 2. MISS DE CACHÉ: Si no existe, buscamos las coordenadas para consultar la API
        if origen not in self.coordenadas_paraderos or destino not in self.coordenadas_paraderos:
            print(f"❌ Error: No se tienen las coordenadas GPS de {origen} o {destino}.")
            return None

        print(f"🌐 [API Request] Descargando datos de tráfico real para {origen} -> {destino}...")
        lat_origen, lon_origen = self.coordenadas_paraderos[origen]
        lat_destino, lon_destino = self.coordenadas_paraderos[destino]

        # URL de OSRM (Cuidado con el orden: OSRM exige Longitud,Latitud)
        url = f"http://router.project-osrm.org/route/v1/driving/{lon_origen},{lat_origen};{lon_destino},{lat_destino}?overview=false"

        try:
            respuesta = requests.get(url)
            if respuesta.status_code == 200:
                data = respuesta.json()
                if data.get('code') == 'Ok':
                    # OSRM devuelve la distancia en metros y el tiempo en segundos.
                    # Lo convertimos a Kilómetros y Minutos para nuestra IA.
                    distancia_km = round(data['routes'][0]['distance'] / 1000, 2)
                    tiempo_minutos = round(data['routes'][0]['duration'] / 60, 2)

                    # Identificamos qué micros hacen este trayecto
                    llave_tramo = self.grafo._generar_llave(origen, destino)
                    servicios = self.servicios_tramos.get(llave_tramo, ["Caminando/Genérico"])

                    # 3. GUARDAMOS EN EL CACHÉ: Para no tener que consultar esto de nuevo
                    self.grafo.registrar_ruta(origen, destino, distancia_km, tiempo_minutos, servicios)

                    return {
                        "distancia_km": distancia_km,
                        "tiempo_minutos": tiempo_minutos,
                        "servicios": servicios
                    }
            print("❌ OSRM no pudo encontrar una ruta viable entre esos puntos.")
        except Exception as e:
            print(f"❌ Error fatal de conexión a internet: {e}")

        return None

Writing motor_logico.py


In [5]:
%%writefile motor_ia.py
import heapq
import itertools
from transformers import pipeline

class MotorIA:
    def __init__(self, motor_logico, hf_token=None):
        self.motor = motor_logico
        print("⏳ [Hugging Face Local] Descargando pesos neuronales a la RAM...")
        print("   (Esto tomará unos 2 minutos solo la primera vez).")

        # Descargamos el modelo TinyLlama directamente desde el HUB de Hugging Face a nuestra RAM
        self.generador = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
        print("✅ [Hugging Face Local] ¡IA cargada e instanciada con éxito!")

    # ==========================================
    # LÓGICA MATEMÁTICA (DIJKSTRA)
    # ==========================================
    def calcular_ruta_optima(self, origen, destino, criterio="tiempo_minutos", es_destino_final=True):
        grafo = self.motor.grafo

        if origen not in grafo.paraderos or destino not in grafo.paraderos:
            return None, "Error: Origen o destino no registrados."

        costos = {nodo: float('inf') for nodo in grafo.paraderos}
        previos = {nodo: None for nodo in grafo.paraderos}
        costos[origen] = 0
        pq = [(0, origen)]

        while pq:
            costo_actual, actual = heapq.heappop(pq)
            if actual == destino: break
            if costo_actual > costos[actual]: continue

            for vecino in grafo.obtener_vecinos(actual):
                datos_arista = self.motor.procesar_ruta(actual, vecino)
                if not datos_arista: continue
                nuevo_costo = costo_actual + datos_arista[criterio]

                if nuevo_costo < costos[vecino]:
                    costos[vecino] = nuevo_costo
                    previos[vecino] = actual
                    heapq.heappush(pq, (nuevo_costo, vecino))

        if costos[destino] == float('inf'): return None, "Ruta imposible."

        # Reconstruimos los paraderos por los que pasamos
        ruta_nodos = []
        paso = destino
        while paso is not None:
            ruta_nodos.insert(0, paso)
            paso = previos[paso]


        # Ahora que sabemos el camino, sumamos ambas cosas sin importar el criterio
        total_tiempo = 0.0
        total_dist_km = 0.0

        for i in range(len(ruta_nodos) - 1):
            datos = self.motor.grafo.consultar_cache(ruta_nodos[i], ruta_nodos[i+1])
            if datos:
                total_tiempo += datos["tiempo_minutos"]
                total_dist_km += datos["distancia_km"]
        # --------------------------------------------------------

        return {
            "ruta_nodos": ruta_nodos,
            "total_tiempo_minutos": round(total_tiempo, 2), # Aquí devolvemos el total exacto
            "total_distancia_km": round(total_dist_km, 2),  # Aquí devolvemos el total exacto
            "itinerario": self._generar_itinerario(ruta_nodos, es_destino_final)
        }, "Éxito"

    def _generar_itinerario(self, ruta_nodos, es_destino_final=True):
        if not ruta_nodos or len(ruta_nodos) < 2:
            return []

        itinerario_humano = []
        micro_actual = None

        for i in range(len(ruta_nodos) - 1):
            origen_tramo = ruta_nodos[i]
            destino_tramo = ruta_nodos[i+1]

            datos = self.motor.grafo.consultar_cache(origen_tramo, destino_tramo)
            micros_disponibles = datos.get("servicios", ["Caminando"]) if datos else ["Caminando"]

            if micro_actual not in micros_disponibles:
                micro_actual = micros_disponibles[0]
                if i == 0:
                    itinerario_humano.append(f"🟢 SUBIR: En paradero {origen_tramo}, toma la opción [{micro_actual}].")
                else:
                    itinerario_humano.append(f"🔄 TRANSBORDO: Bájate en {origen_tramo} y cambia a [{micro_actual}].")

            itinerario_humano.append(f"   -> Viajando hacia {destino_tramo}...")

        if es_destino_final:
            itinerario_humano.append(f"🔴 BAJAR: Llegaste a tu destino en {ruta_nodos[-1]}.")
        else:
            itinerario_humano.append(f"📍 PARADA INTERMEDIA: Llegaste a {ruta_nodos[-1]}.")

        return itinerario_humano

    # ==========================================
    # MÚLTIPLES PARADAS (hasta 2 paradas intermedias opcionales)
    # Prueba todos los órdenes posibles entre las paradas seleccionadas
    # y se queda con el de menor costo según el criterio pedido.
    # ==========================================
    def calcular_ruta_multiparada(self, origen, destino, paradas_intermedias, criterio="tiempo_minutos"):
        """
        Calcula la ruta óptima pasando obligatoriamente por las paradas_intermedias dadas
        (0, 1 o 2 paradas). Prueba cada orden posible entre ellas y devuelve el mejor.
        """
        # Filtramos vacíos/None, duplicados y paradas iguales al origen/destino
        paradas_validas = list(dict.fromkeys(
            p for p in paradas_intermedias if p and p not in (origen, destino)
        ))

        mejor_resultado = None
        mejor_costo = float('inf')

        # Con 0, 1 o 2 paradas intermedias esto son como máximo 2 combinaciones (2!)
        for orden in itertools.permutations(paradas_validas):
            secuencia = [origen] + list(orden) + [destino]
            tiempo_total = 0.0
            distancia_total = 0.0
            itinerario_completo = []
            ruta_nodos_completa = []
            valido = True

            for i in range(len(secuencia) - 1):
                sub_origen = secuencia[i]
                sub_destino = secuencia[i + 1]
                es_ultimo_tramo = (i == len(secuencia) - 2)

                resultado, _ = self.calcular_ruta_optima(
                    sub_origen, sub_destino, criterio, es_destino_final=es_ultimo_tramo
                )

                if not resultado:
                    valido = False
                    break

                tiempo_total += resultado["total_tiempo_minutos"]
                distancia_total += resultado["total_distancia_km"]

                # Evitamos duplicar el nodo de unión entre tramos consecutivos
                nodos_tramo = resultado["ruta_nodos"]
                if ruta_nodos_completa and nodos_tramo and ruta_nodos_completa[-1] == nodos_tramo[0]:
                    ruta_nodos_completa.extend(nodos_tramo[1:])
                else:
                    ruta_nodos_completa.extend(nodos_tramo)

                itinerario_completo.extend(resultado["itinerario"])

            if not valido:
                continue

            costo_total = tiempo_total if criterio == "tiempo_minutos" else distancia_total

            if costo_total < mejor_costo:
                mejor_costo = costo_total
                mejor_resultado = {
                    "ruta_nodos": ruta_nodos_completa,
                    "total_tiempo_minutos": round(tiempo_total, 2),
                    "total_distancia_km": round(distancia_total, 2),
                    "itinerario": itinerario_completo,
                    "orden_paradas": list(orden)
                }

        if mejor_resultado is None:
            return None, "Ruta imposible: no se pudo conectar el origen, las paradas intermedias y el destino."

        return mejor_resultado, "Éxito"

    # ==========================================
    # LÓGICA COGNITIVA (IA LOCAL DE HUGGING FACE)
    # ==========================================
    def evaluar_estrategia_logistica(self, origen, destino, contexto_usuario):
        """Orquestador principal. Usa la IA que descargamos en la memoria RAM."""
        ruta_tiempo, _ = self.calcular_ruta_optima(origen, destino, "tiempo_minutos")
        ruta_distancia, _ = self.calcular_ruta_optima(origen, destino, "distancia_km")

        if not ruta_tiempo or not ruta_distancia:
            return "❌ No se pudo calcular la ruta. Verifica el GTFS."

        prompt_analisis = f"""[INST] Eres un experto Analista Logístico de transporte. Responde en un párrafo corto.
Cliente viaja de {origen} a {destino}.
Su contexto es: "{contexto_usuario}".

OPCIÓN A (Rápida): {ruta_tiempo.get('total_tiempo_minutos')} min, {ruta_tiempo.get('total_distancia_km')} km.
OPCIÓN B (Corta): {ruta_distancia.get('total_tiempo_minutos')} min, {ruta_distancia.get('total_distancia_km')} km.

Recomienda cuál debe tomar justificando tu respuesta de acuerdo a su contexto. [/INST]"""

        # Formato de "Prompt" especial que exige TinyLlama para pensar
        prompt_formateado = f"<|system|>\nEres un analista logístico útil y conciso. Responde en español.</s>\n<|user|>\n{prompt_analisis}</s>\n<|assistant|>\n"

        try:
            # Aquí le decimos a nuestro modelo local que escriba la respuesta
            resultado = self.generador(prompt_formateado, max_new_tokens=200, temperature=0.3, return_full_text=False)
            texto_generado = resultado[0]['generated_text']

            informe_final = f"### 🤖 Análisis Logístico (IA Local Hugging Face)\n{texto_generado}\n\n"
            informe_final += f"**Ruta Sugerida (Matemática):**\n"
            informe_final += chr(10).join(['* ' + paso for paso in ruta_tiempo.get('itinerario', [])])
            return informe_final

        except Exception as e:
            return f"❌ Error en la IA Local: {e}"

    def evaluar_estrategia_logistica_multiparada(self, origen, parada_1, parada_2, destino, contexto_usuario):
        """
        Igual que evaluar_estrategia_logistica, pero acepta hasta 2 paradas intermedias
        opcionales. Prueba todos los órdenes posibles entre las paradas seleccionadas
        y usa el de menor costo para cada criterio (tiempo y distancia).
        """
        paradas_intermedias = [p for p in (parada_1, parada_2) if p and p != "Ninguna"]

        ruta_tiempo, msg_tiempo = self.calcular_ruta_multiparada(origen, destino, paradas_intermedias, "tiempo_minutos")
        ruta_distancia, _ = self.calcular_ruta_multiparada(origen, destino, paradas_intermedias, "distancia_km")

        if not ruta_tiempo or not ruta_distancia:
            return f"❌ No se pudo calcular la ruta con esas paradas. ({msg_tiempo})"

        paradas_texto = " → ".join(paradas_intermedias) if paradas_intermedias else "Sin paradas intermedias (viaje directo)"
        orden_tiempo_texto = " → ".join(ruta_tiempo.get('orden_paradas', [])) or "directo"
        orden_distancia_texto = " → ".join(ruta_distancia.get('orden_paradas', [])) or "directo"

        prompt_analisis = f"""[INST] Eres un experto Analista Logístico de transporte. Responde en un párrafo corto.
Cliente viaja de {origen} a {destino}, pasando obligatoriamente por: {paradas_texto}.
Su contexto es: "{contexto_usuario}".

OPCIÓN A (Rápida): orden de paradas {orden_tiempo_texto}, {ruta_tiempo.get('total_tiempo_minutos')} min, {ruta_tiempo.get('total_distancia_km')} km.
OPCIÓN B (Corta): orden de paradas {orden_distancia_texto}, {ruta_distancia.get('total_tiempo_minutos')} min, {ruta_distancia.get('total_distancia_km')} km.

Recomienda cuál debe tomar justificando tu respuesta de acuerdo a su contexto. [/INST]"""

        prompt_formateado = f"<|system|>\nEres un analista logístico útil y conciso. Responde en español.</s>\n<|user|>\n{prompt_analisis}</s>\n<|assistant|>\n"

        try:
            resultado = self.generador(prompt_formateado, max_new_tokens=200, temperature=0.3, return_full_text=False)
            texto_generado = resultado[0]['generated_text']

            informe_final = f"### 🤖 Análisis Logístico (IA Local Hugging Face)\n{texto_generado}\n\n"
            informe_final += f"**Paradas intermedias seleccionadas:** {paradas_texto}\n\n"
            informe_final += f"**Ruta Sugerida (Matemática, orden: {orden_tiempo_texto}):**\n"
            informe_final += chr(10).join(['* ' + paso for paso in ruta_tiempo.get('itinerario', [])])
            return informe_final

        except Exception as e:
            return f"❌ Error en la IA Local: {e}"


Writing motor_ia.py


In [ ]:
import pandas as pd
import json
import os

# 1. Crear la carpeta data si no existe (por si corres esto en un Colab nuevo)
os.makedirs("../data", exist_ok=True)

def cargar_paraderos_desde_arcgis(ruta_archivo_csv):
    """Lee el dataset geoespacial y lo transforma a un diccionario."""
    print(f"📂 Cargando dataset geoespacial desde: {"~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/Paraderos_Transantiago.csv"}...")

    try:
        df = pd.read_csv("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/Paraderos_Transantiago.csv", sep=',')
        diccionario_paraderos = {}

        for index, fila in df.iterrows():
            id_parada = str(fila['simt'])
            latitud = float(fila['x'])
            longitud = float(fila['y'])
            diccionario_paraderos[id_parada] = (latitud, longitud)

        print(f"✅ Éxito: Se procesaron {len(diccionario_paraderos)} paraderos.")
        return diccionario_paraderos

    except FileNotFoundError:
        print("❌ Error: No se encontró el archivo CSV.")
        return {}
    except KeyError as e:
        print(f"❌ Error: No existe la columna {e} en tu CSV.")
        return {}

# ==========================================
# EJECUCIÓN Y EXPORTACIÓN A JSON
# ==========================================
# Nota: Usamos "../data/" porque asumimos que este notebook está dentro de la carpeta "notebooks/"
ruta_entrada = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/Paraderos_Transantiago.csv"
ruta_salida = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/arcgis_limpio.json"

paraderos_procesados = cargar_paraderos_desde_arcgis(ruta_entrada)

if paraderos_procesados:
    print("💾 Exportando a JSON...")
    # Empaquetamos en un diccionario maestro por estándar
    datos_exportar = {"paraderos": paraderos_procesados}

    with open(ruta_salida, "w", encoding="utf-8") as archivo:
        json.dump(datos_exportar, archivo, ensure_ascii=False, indent=4)

    print(f"🚀 ¡Listo! Archivo guardado en: {"~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/arcgis_limpio.json"}")

In [ ]:
import pandas as pd
import json
import os

# 1. Asegurar la existencia de la carpeta
os.makedirs("../data", exist_ok=True)

def construir_grafo_desde_gtfs(ruta_stops, ruta_stop_times):
    print("📊 [1/3] Extrayendo Vértices desde stops.txt...")
    df_stops = pd.read_csv("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stops.txt", sep=',')

    mis_paraderos = {}
    for _, fila in df_stops.iterrows():
        mis_paraderos[str(fila['stop_id'])] = (float(fila['stop_lat']), float(fila['stop_lon']))

    print("🚌 [2/3] Procesando Aristas desde stop_times.txt (Big Data)...")
    df_times = pd.read_csv("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stop_times.txt", usecols=['trip_id', 'stop_id', 'stop_sequence'], sep=',')
    df_times = df_times.sort_values(by=['trip_id', 'stop_sequence'])

    trip_ids = df_times['trip_id'].values
    stop_ids = df_times['stop_id'].values

    diccionario_servicios = {}
    adyacencias_grafo = {nodo: set() for nodo in mis_paraderos.keys()}

    print("⚙️ [3/3] Tejiendo la Red Vectorial del Grafo...")
    for i in range(len(trip_ids) - 1):
        viaje_actual = trip_ids[i]
        viaje_siguiente = trip_ids[i + 1]

        if viaje_actual == viaje_siguiente:
            origen = str(stop_ids[i])
            destino = str(stop_ids[i + 1])
            micro = str(viaje_actual).split('-')[0]

            llave_tramo = f"{origen}-{destino}"
            if llave_tramo not in diccionario_servicios:
                diccionario_servicios[llave_tramo] = set()
            diccionario_servicios[llave_tramo].add(micro)

            if origen in adyacencias_grafo:
                adyacencias_grafo[origen].add(destino)

    # Limpieza Final para compatibilidad con JSON
    diccionario_servicios = {k: list(v) for k, v in diccionario_servicios.items()}
    adyacencias_grafo = {k: list(v) for k, v in adyacencias_grafo.items()}

    print(f"✅ ¡GTFS LISTO! {len(mis_paraderos)} paraderos y {len(diccionario_servicios)} calles.")
    return mis_paraderos, diccionario_servicios, adyacencias_grafo

# ==========================================
# EJECUCIÓN Y EXPORTACIÓN A JSON
# ==========================================
ruta_stops = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stops.txt"
ruta_times = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stop_times.txt"
ruta_salida_gtfs = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/gtfs_limpio.json"

paraderos, servicios, conexiones = construir_grafo_desde_gtfs(ruta_stops, ruta_times)

print("💾 Exportando a JSON...")
# Empaquetamos todo el Grafo en un solo archivo JSON
datos_exportar = {
    "paraderos": paraderos,
    "servicios": servicios,
    "conexiones": conexiones
}

with open(ruta_salida_gtfs, "w", encoding="utf-8") as archivo:
    json.dump(datos_exportar, archivo, ensure_ascii=False, indent=4)

print(f"🚀 ¡Listo! Grafo completo guardado en: {"~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/gtfs_limpio.json"}")

In [8]:
# @title
import gradio as gr
from google.colab import userdata
from grafo_logistica import GrafoLogistica
from motor_logico import MotorLogico
from motor_ia import MotorIA

# 1. Recuperar token y cargar datos
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = None

# Ya no usamos cargar_paraderos_desde_arcgis. Usamos los datos procesados en la celda anterior.

grafo_memoria = GrafoLogistica()

# 🚨 Inyección de Vértices y Aristas directa al núcleo del Grafo
grafo_memoria.paraderos = set(paraderos_reales.keys())
grafo_memoria.adyacencia = conexiones_reales

# Iniciamos el Motor de Red
motor_red = MotorLogico(grafo_memoria)
motor_red.cargar_datos_base(paraderos_reales, servicios_reales)

# Iniciamos la IA
cerebro_ia = MotorIA(motor_red, HF_TOKEN)

# ==========================================
# 5. INTERFAZ GRÁFICA (Gradio)
# ==========================================
# Cuidado: Como ahora tenemos miles de paraderos, el Dropdown puede ser pesado.
# Seleccionamos solo los primeros 1000 para que la interfaz web cargue rápido.
nombres_paraderos = list(paraderos_reales.keys())[:1000]

# Para las paradas intermedias (opcionales) agregamos la opción "Ninguna" al principio
opciones_intermedias = ["Ninguna"] + nombres_paraderos

# Fíjate qué limpio: Gradio llama DIRECTAMENTE a cerebro_ia.evaluar_estrategia_logistica_multiparada
# El sistema prueba automáticamente todos los órdenes posibles entre las paradas
# intermedias seleccionadas y se queda con el de menor costo.
app = gr.Interface(
    fn=cerebro_ia.evaluar_estrategia_logistica_multiparada,
    inputs=[
        gr.Dropdown(choices=nombres_paraderos, label="📍 Origen"),
        gr.Dropdown(choices=opciones_intermedias, value="Ninguna", label="➕ Parada intermedia 1 (opcional)"),
        gr.Dropdown(choices=opciones_intermedias, value="Ninguna", label="➕ Parada intermedia 2 (opcional)"),
        gr.Dropdown(choices=nombres_paraderos, label="🎯 Destino"),
        gr.Textbox(lines=2, label="🗣️ Contexto")
    ],
    outputs=gr.Markdown(label="🧠 Decisión del Agente"),
    title="🤖 Agente Logístico",
)

app.launch(share=True)


⏳ [Hugging Face Local] Descargando pesos neuronales a la RAM...
   (Esto tomará unos 2 minutos solo la primera vez).


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ [Hugging Face Local] ¡IA cargada e instanciada con éxito!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf1c447bf7207d25d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
